# Work efficiently and reproducibly · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 23. Work efficiently and reproducibly

Read less data, combine chunk summaries correctly, and make repeatable samples.

Before you start: Understand grouping, CSV I/O, and data types.

Official reference: https://pandas.pydata.org/docs/user_guide/scale.html

### M23C01. Measure memory use

memory_usage(deep=True) estimates memory including text stored by object columns. Repeated labels may use less memory as categories. The benefit depends on the data, so measure it instead of assuming.

**Read:** Why repeat only two labels many times?

Your explanation: _write here_.

**Predict:** Will converting to category change the label values?

Your explanation: _write here_.

**Explain:** Are exact byte counts guaranteed on every computer?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will converting to category change the label values?
labels = pd.Series(["RAK", "Dubai"] * 500, dtype=object)
category = labels.astype("category")
result = pd.Series({"Object_bytes": labels.memory_usage(deep=True), "Category_bytes": category.memory_usage(deep=True)})
print(result)

**Your turn:** Measure 1,000 unique text labels before and after category conversion. Explain why the saving may shrink or disappear.

For the check, keep these result names: `category`, `labels`.

**Hint:** Build labels such as "ID_0" through "ID_999".

**Watch for this:** This is a small storage demonstration, not a benchmark of all pandas workloads.

In [ ]:
# Change this code to complete the task.
# Question: Measure 1,000 unique text labels before and after category conversion. Explain why the saving may shrink or disappear.
labels = pd.Series(["RAK", "Dubai"] * 500, dtype=object)
category = labels.astype("category")
result = pd.Series({"Object_bytes": labels.memory_usage(deep=True), "Category_bytes": category.memory_usage(deep=True)})
print(result)

**Quick check:** When is category often useful for memory?

1. Many repeated values from a small set
2. Every value is unique, always
3. Only when the column is numeric

**Explain what you learned:** Choose data types based on meaning and measurement.

### M23C02. Load only needed columns

usecols avoids reading columns that the question does not need. Explicit dtypes make the import contract clearer. This small file demonstrates a habit that becomes useful with larger files.

**Read:** Which six source columns are omitted?

Your explanation: _write here_.

**Predict:** How many rows are loaded?

Your explanation: _write here_.

**Explain:** Why not load every field just in case?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many rows are loaded?
result = pd.read_csv(DATA_DIR / "students.csv", usecols=["Student_ID", "Program", "Score"], dtype={"Student_ID": "string", "Program": "category"})
print(result.head())
print(result.dtypes)

**Your turn:** Read only Student_ID and Attendance, and limit the preview to the first four file rows with nrows.

For the check, keep these result names: `result`.

**Hint:** Use nrows=4 inside read_csv.

**Watch for this:** A file preview can miss problems that occur later in the file.

In [ ]:
# Change this code to complete the task.
# Question: Read only Student_ID and Attendance, and limit the preview to the first four file rows with nrows.
result = pd.read_csv(DATA_DIR / "students.csv", usecols=["Student_ID", "Program", "Score"], dtype={"Student_ID": "string", "Program": "category"})
print(result.head())
print(result.dtypes)

**Quick check:** What does nrows=4 do?

1. Loads only the first four data rows
2. Chooses four random rows
3. Loads four columns

**Explain what you learned:** Read the fields and rows needed for the current step.

### M23C03. Combine chunk totals correctly

chunksize reads a CSV in pieces. A total and a non-missing count can be accumulated across pieces without keeping all rows. Divide the overall total by the overall count to calculate the overall mean.

**Read:** What is chunk in each loop iteration?

Your explanation: _write here_.

**Predict:** How many chunks are processed?

Your explanation: _write here_.

**Explain:** Why not average the three chunk means equally?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many chunks are processed?
total = 0
count = 0
for chunk in pd.read_csv(DATA_DIR / "students.csv", usecols=["Score"], chunksize=5):
    total += chunk["Score"].sum()
    count += chunk["Score"].count()
result = total / count
print("Known scores:", count)
print("Mean score:", round(result, 2))

**Your turn:** Use chunks of four rows and calculate how many students score at least 80 without retaining all rows.

For the check, keep these result names: `result`.

**Hint:** Add the sum of a Boolean condition for each chunk.

**Watch for this:** Not every operation is easy to compute in chunks. A global sort or median needs a different strategy.

In [ ]:
# Change this code to complete the task.
# Question: Use chunks of four rows and calculate how many students score at least 80 without retaining all rows.
total = 0
count = 0
for chunk in pd.read_csv(DATA_DIR / "students.csv", usecols=["Score"], chunksize=5):
    total += chunk["Score"].sum()
    count += chunk["Score"].count()
result = total / count
print("Known scores:", count)
print("Mean score:", round(result, 2))

**Quick check:** What is needed for an exact combined mean?

1. An unweighted mean of chunk means
2. A combined sum and non-missing count
3. Only the last chunk

**Explain what you learned:** Combine summaries using their correct mathematical weights.

### M23C04. Take a repeatable sample

sample chooses rows randomly. random_state makes the selection repeatable for this data and environment. Reproducibility does not make a tiny sample representative of the population.

**Read:** What does n=3 request?

Your explanation: _write here_.

**Predict:** Will the repeated call with the same input and seed match?

Your explanation: _write here_.

**Explain:** Does a fixed seed remove sampling bias?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will the repeated call with the same input and seed match?
df = students()
result = df.sample(n=3, random_state=7)
repeat = df.sample(n=3, random_state=7)
print(result[["Student_ID", "Campus", "Score"]])
print("Same sample:", result.equals(repeat))

**Your turn:** Take two students from each campus using groupby.sample with random_state=7.

For the check, keep these result names: `result`.

**Hint:** Group by Campus before calling sample(n=2, random_state=7).

**Watch for this:** Document input data, row order, library versions, and seed when an exact sample matters.

In [ ]:
# Change this code to complete the task.
# Question: Take two students from each campus using groupby.sample with random_state=7.
df = students()
result = df.sample(n=3, random_state=7)
repeat = df.sample(n=3, random_state=7)
print(result[["Student_ID", "Campus", "Score"]])
print("Same sample:", result.equals(repeat))

**Quick check:** What does a seed mainly provide?

1. Guaranteed representativeness
2. Repeatability under the same setup
3. Higher scores

**Explain what you learned:** Repeatability and representativeness are different goals.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates